# 02. Transformasi & Pipeline ETL PySpark
**Studi Kasus 08: Tourism Analytics**

Notebook ini mengimplementasikan transformasi data berskala besar menggunakan **PySpark** dengan data yang diunduh langsung dari Kaggle menggunakan **kagglehub**.

In [1]:
# ==============================================================================
# CELL 1: Inisialisasi PySpark Session & Import Modules
# ==============================================================================
import os
import kagglehub
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, expr, count, sum as _sum, avg, round, coalesce, lit,
    concat_ws, row_number, rank
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType

path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")
csv_path = os.path.join(path, "hotel_bookings.csv")
if not os.path.exists(csv_path):
    csv_path = '../data/raw/hotel_bookings.csv'

spark = SparkSession.builder \
    .appName("TourismAnalyticsETL_Advanced") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("PySpark Session Siap Digunakan.")

# ==============================================================================
# CELL 2: Data Ingestion & Quality Checks
# ==============================================================================
df_spark = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "NA") \
    .csv(csv_path)

# Enforce Explicit Type Casting
int_cols = ['children', 'adults', 'babies', 'stays_in_weekend_nights',
            'stays_in_week_nights', 'is_canceled', 'lead_time']
for c in int_cols:
    df_spark = df_spark.withColumn(c, col(c).cast(IntegerType()))

df_spark = df_spark.withColumn('adr', col('adr').cast(DoubleType()))

# Data Cleaning & Filter Anomali Bisnis
df_clean = df_spark \
    .na.fill({'children': 0, 'adults': 0, 'babies': 0, 'country': 'Unknown', 'agent': '0', 'company': '0'}) \
    .filter(col('adr') >= 0) \
    .filter((col('adults') + col('children') + col('babies')) > 0)

print(f"Total Baris Data Terisi & Bersih: {df_clean.count():,}")

# ==============================================================================
# CELL 3: Feature Engineering Komprehensif
# ==============================================================================
df_transformed = df_clean \
    .withColumn("total_nights", col("stays_in_weekend_nights") + col("stays_in_week_nights")) \
    .withColumn("total_guests", col("adults") + col("children") + col("babies")) \
    .withColumn("is_family", when((col("children") > 0) | (col("babies") > 0), 1).otherwise(0)) \
    .withColumn("lead_time_category",
                when(col("lead_time") <= 14, "Short (<2 Weeks)")
                .when((col("lead_time") > 14) & (col("lead_time") <= 60), "Medium (2-8 Weeks)")
                .otherwise("Long (>2 Months)")) \
    .withColumn("gross_revenue", round(col("adr") * col("total_nights"), 2)) \
    .withColumn("net_revenue", when(col("is_canceled") == 0, round(col("adr") * col("total_nights"), 2)).otherwise(0.0)) \
    .withColumn("status_desc", when(col("is_canceled") == 1, "Canceled").otherwise("Check-Out"))

df_transformed.select("hotel", "total_guests", "lead_time_category", "gross_revenue", "net_revenue", "status_desc").show(5)

# ==============================================================================
# CELL 4: Agregasi Multi-Dimensi & Window Functions
# ==============================================================================
# 1. Agregasi Bulanan Performance Hotel
monthly_summary = df_transformed.groupBy("hotel", "arrival_date_year", "arrival_date_month") \
    .agg(
        count("*").alias("total_bookings"),
        _sum("net_revenue").alias("total_net_revenue"),
        round(avg("adr"), 2).alias("avg_daily_rate"),
        round(avg("is_canceled") * 100, 2).alias("cancellation_rate_pct")
    )

# 2. Window Function: Ranking Bulan Berdasarkan Revenue per Hotel
window_spec = Window.partitionBy("hotel", "arrival_date_year").orderBy(col("total_net_revenue").desc())
monthly_ranked = monthly_summary.withColumn("rank_revenue_month", rank().over(window_spec))

print("=== SAMPLE PERFORMA BULANAN TERBAIK ===")
monthly_ranked.filter(col("rank_revenue_month") <= 3).show(10)

# ==============================================================================
# CELL 5: Export Data Hasil Transformasi Ke Format Parquet
# ==============================================================================
output_dir = "../data/processed/parquet_output"
os.makedirs(output_dir, exist_ok=True)
parquet_path = os.path.join(output_dir, "hotel_bookings_clean.parquet")

df_transformed.write \
    .mode("overwrite") \
    .partitionBy("hotel") \
    .parquet(parquet_path)

print(f"Data ETL PySpark berhasil disimpan dalam format Parquet terpartisi di: {parquet_path}")

Using Colab cache for faster access to the 'hotel-booking-demand' dataset.
PySpark Session Siap Digunakan.
Total Baris Data Terisi & Bersih: 119,209
+------------+------------+------------------+-------------+-----------+-----------+
|       hotel|total_guests|lead_time_category|gross_revenue|net_revenue|status_desc|
+------------+------------+------------------+-------------+-----------+-----------+
|Resort Hotel|           2|  Long (>2 Months)|          0.0|        0.0|  Check-Out|
|Resort Hotel|           2|  Long (>2 Months)|          0.0|        0.0|  Check-Out|
|Resort Hotel|           1|  Short (<2 Weeks)|         75.0|       75.0|  Check-Out|
|Resort Hotel|           1|  Short (<2 Weeks)|         75.0|       75.0|  Check-Out|
|Resort Hotel|           2|  Short (<2 Weeks)|        196.0|      196.0|  Check-Out|
+------------+------------+------------------+-------------+-----------+-----------+
only showing top 5 rows
=== SAMPLE PERFORMA BULANAN TERBAIK ===
+------------+--------